# Patient-Level Benchmarks

This notebook shows, in executable code, how the public patient-level benchmark numbers are computed.

It intentionally does **not** just load the released metric summaries. Instead it:

1. Loads the public patient score/readout artifacts.
2. Reimplements the score formulas and metric functions directly in the notebook.
3. Recomputes CRC, cSCC, and CRC module-cosine metrics from row-level data.
4. Checks the recomputed values against the checked-in metric CSVs.
5. Generates the patient-level figures under `artifacts/notebook_figures/`.


In [ ]:
# Standard-library imports only. The notebook is intentionally self-contained.
# We do not import the package-level reproduce_* helpers for the calculations below.
from pathlib import Path
import csv
import html
import json
import math

# IPython display is optional. The notebook still runs as plain Python.
try:
    from IPython.display import SVG, Markdown, display
except Exception:
    SVG = Markdown = None
    def display(value):
        print(value)

# Find the repository root from wherever Jupyter launched this notebook.
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Run this notebook from inside the open-benchmarks repo")
    ROOT = ROOT.parent

# Generated figures are reproducible outputs and are intentionally gitignored.
FIG_DIR = ROOT / "artifacts/notebook_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Public patient artifact locations used by this notebook.
CRC_DIR = ROOT / "patient-level-bench/model_scores/crc_moa_tailored_20260525"
CSCC_DIR = ROOT / "patient-level-bench/model_scores/cscc_checkpoint_compartment_20260604"
COSINE_DIR = ROOT / "patient-level-bench/observed_readouts/crc_module_mean_cosine_20260604"
CRC_ROWS_CSV = CRC_DIR / "crc_patient_moa_tailored_rank_scores_20260525.csv"
CRC_METRICS_CSV = CRC_DIR / "crc_patient_moa_tailored_metrics_20260525.csv"
CSCC_ROWS_CSV = CSCC_DIR / "cscc_checkpoint_compartment_patient_scores_20260604.csv"
CSCC_METRICS_CSV = CSCC_DIR / "cscc_checkpoint_compartment_metrics_20260604.csv"
COSINE_MODULE_VECTORS_CSV = COSINE_DIR / "crc_module_mean_cosine_module_vectors.csv"
COSINE_PATIENT_STEPS_CSV = COSINE_DIR / "crc_module_mean_cosine_patient_steps.csv"
COSINE_STEP_SUMMARY_CSV = COSINE_DIR / "crc_module_mean_cosine_step_summary.csv"

print(f"Repository root: {ROOT}")
print(f"Figures will be written to: {FIG_DIR.relative_to(ROOT)}")


## Shared CSV and Metric Code

These functions are the benchmark machinery. They are written out here so a reader can see exactly how patient AUC, balanced accuracy, Spearman, quantiles, and cosine readouts are computed.

In [ ]:
# -----------------------------------------------------------------------------
# Shared metric helpers for patient-level benchmarks
# -----------------------------------------------------------------------------
# This cell is the patient benchmark engine used below. It intentionally
# redefines metrics and simple transforms in the notebook so a reviewer can see
# exactly how the patient-level numbers are computed from CSV rows.
#
# Important patient convention:
#   score columns are assigned without using labels.
#   labels are used only after scoring to compute AUC, balanced accuracy,
#   Spearman, and responder/non-responder means.
# -----------------------------------------------------------------------------

def read_csv_rows(path):
    """Read a CSV file as a list of dictionaries, preserving all public columns."""
    with Path(path).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


def finite_float(value, default=math.nan):
    """Convert CSV strings to floats and return NaN for missing/non-finite values."""
    try:
        out = float(value)
    except (TypeError, ValueError):
        return default
    return out if math.isfinite(out) else default


def mean(values):
    """Mean over finite numeric values."""
    clean = [finite_float(value) for value in values]
    clean = [value for value in clean if math.isfinite(value)]
    return sum(clean) / len(clean) if clean else math.nan


def quantile(values, q):
    """Linear-interpolated quantile over finite numeric values."""
    clean = sorted(finite_float(value) for value in values)
    clean = [value for value in clean if math.isfinite(value)]
    if not clean:
        return math.nan
    if len(clean) == 1:
        return clean[0]
    pos = q * (len(clean) - 1)
    lo = int(math.floor(pos))
    hi = int(math.ceil(pos))
    if lo == hi:
        return clean[lo]
    frac = pos - lo
    return clean[lo] * (1.0 - frac) + clean[hi] * frac


def finite_pairs(xs, ys):
    """Keep only paired finite values before computing a metric."""
    pairs = []
    for x_raw, y_raw in zip(xs, ys):
        x = finite_float(x_raw)
        y = finite_float(y_raw)
        if math.isfinite(x) and math.isfinite(y):
            pairs.append((x, y))
    return pairs


def pearson(xs, ys):
    """Ordinary Pearson correlation on finite paired values."""
    pairs = finite_pairs(xs, ys)
    if len(pairs) < 2:
        return math.nan
    x_values = [x for x, _ in pairs]
    y_values = [y for _, y in pairs]
    x_mean = mean(x_values)
    y_mean = mean(y_values)
    x_ss = sum((x - x_mean) ** 2 for x in x_values)
    y_ss = sum((y - y_mean) ** 2 for y in y_values)
    if x_ss <= 0.0 or y_ss <= 0.0:
        return math.nan
    cov = sum((x - x_mean) * (y - y_mean) for x, y in pairs)
    return cov / math.sqrt(x_ss * y_ss)


def rankdata(values):
    """Average ranks, 1-indexed, with ties handled like scipy.stats.rankdata."""
    order = sorted(range(len(values)), key=lambda idx: values[idx])
    ranks = [0.0] * len(values)
    start = 0
    while start < len(order):
        end = start
        while end + 1 < len(order) and values[order[end + 1]] == values[order[start]]:
            end += 1
        rank = (start + 1 + end + 1) / 2.0
        for pos in range(start, end + 1):
            ranks[order[pos]] = rank
        start = end + 1
    return ranks


def spearman(xs, ys):
    """Spearman rho as Pearson correlation of average ranks."""
    pairs = finite_pairs(xs, ys)
    if len(pairs) < 2:
        return math.nan
    x_ranks = rankdata([x for x, _ in pairs])
    y_ranks = rankdata([y for _, y in pairs])
    return pearson(x_ranks, y_ranks)


def roc_auc(labels, scores, positive=1):
    """Pairwise AUC: probability a positive row scores above a negative row."""
    pairs = []
    for label_raw, score_raw in zip(labels, scores):
        score = finite_float(score_raw)
        if not math.isfinite(score):
            continue
        try:
            label = int(float(label_raw))
        except (TypeError, ValueError):
            continue
        pairs.append((label, score))
    positives = [score for label, score in pairs if label == positive]
    negatives = [score for label, score in pairs if label != positive]
    if not positives or not negatives:
        return math.nan
    wins = 0.0
    for positive_score in positives:
        for negative_score in negatives:
            if positive_score > negative_score:
                wins += 1.0
            elif positive_score == negative_score:
                wins += 0.5
    return wins / (len(positives) * len(negatives))


def balanced_accuracy(labels, predictions, positive=1):
    """Balanced accuracy: average of sensitivity and specificity."""
    y_true = [int(float(label)) for label in labels]
    y_pred = [int(bool(prediction)) for prediction in predictions]
    positives = [pred for label, pred in zip(y_true, y_pred) if label == positive]
    negatives = [pred for label, pred in zip(y_true, y_pred) if label != positive]
    if not positives or not negatives:
        return math.nan
    sensitivity = sum(pred == positive for pred in positives) / len(positives)
    specificity = sum(pred != positive for pred in negatives) / len(negatives)
    return 0.5 * (sensitivity + specificity)


def sigmoid(value, center=0.0, scale=1.0):
    """Numerically stable logistic transform used by the CRC soft-min gates."""
    numeric = finite_float(value)
    if not math.isfinite(numeric):
        return math.nan
    z = max(-40.0, min(40.0, (numeric - center) / scale))
    return 1.0 / (1.0 + math.exp(-z))


def cosine(xs, ys):
    """Cosine similarity between two finite numeric vectors."""
    pairs = finite_pairs(xs, ys)
    if len(pairs) < 2:
        return math.nan
    x_values = [x for x, _ in pairs]
    y_values = [y for _, y in pairs]
    x_norm = math.sqrt(sum(x * x for x in x_values))
    y_norm = math.sqrt(sum(y * y for y in y_values))
    denom = x_norm * y_norm
    if denom <= 0.0:
        return math.nan
    return sum(x * y for x, y in pairs) / denom


def assert_close(actual, expected, label, tol=1e-9):
    """Fail loudly if a recomputed value drifts from the checked-in release value."""
    actual_value = finite_float(actual)
    expected_value = finite_float(expected)
    if math.isnan(actual_value) and math.isnan(expected_value):
        return
    if not math.isclose(actual_value, expected_value, rel_tol=tol, abs_tol=tol):
        raise AssertionError(f"{label}: actual={actual_value} expected={expected_value}")

print("Metric functions loaded.")


## CRC MOA-Tailored Patient Rank Score

The CRC benchmark starts from five broad module supports, computes a label-free soft-min response intermediate, then turns that intermediate into a panel-relative rank score.

```text
coverage = fraction(module supports > 0)
mean_support = mean(module supports)
p10_support = 10th percentile(module supports)
effective_coverage = min(1, coverage / 0.80)

intermediate = min(
  sigmoid(effective_coverage, center=0.90, scale=0.05),
  sigmoid(mean_support, center=0.0, scale=0.015),
  sigmoid(p10_support, center=-0.02, scale=0.015)
)

response_score_rank_calibrated = rank(intermediate within 11-patient CRC panel) / 11
```

The score assignment uses no labels. Labels are used only after scoring to compute AUC and balanced accuracy.

In [ ]:
# -----------------------------------------------------------------------------
# Benchmark 1: CRC MOA-tailored patient rank score
# -----------------------------------------------------------------------------
# Input artifact:
#   patient-level-bench/model_scores/crc_moa_tailored_20260525/
#   crc_patient_moa_tailored_rank_scores_20260525.csv
#
# Label column:
#   observed_responder
#
# Public score column:
#   response_score_rank_calibrated
#
# What this cell does:
#   1. Recompute broad-module coverage, mean support, p10 support.
#   2. Apply the label-free soft-min response gate.
#   3. Convert the intermediate into an 11-patient panel rank score.
#   4. Use labels only after scoring to compute AUC and balanced accuracy.
# -----------------------------------------------------------------------------

crc_rows = read_csv_rows(CRC_ROWS_CSV)
CRC_SUPPORT_COLS = [
    "kras_mapk_support",
    "egfr_support",
    "cytostasis_support",
    "escape_control_support",
    "kill_conversion_support",
]

# First pass: compute the label-free module-calibrated intermediate per patient.
for row in crc_rows:
    supports = [finite_float(row[col]) for col in CRC_SUPPORT_COLS]
    coverage = sum(value > 0.0 for value in supports) / len(supports)
    mean_support = mean(supports)
    p10_support = quantile(supports, 0.10)
    effective_coverage = min(1.0, coverage / 0.80)
    intermediate = min(
        sigmoid(effective_coverage, center=0.90, scale=0.05),
        sigmoid(mean_support, center=0.0, scale=0.015),
        sigmoid(p10_support, center=-0.02, scale=0.015),
    )

    # Store every intermediate so the row-level calculation is inspectable.
    row["recomputed_coverage"] = coverage
    row["recomputed_mean_support"] = mean_support
    row["recomputed_p10_support"] = p10_support
    row["recomputed_effective_coverage"] = effective_coverage
    row["recomputed_module_calibrated_intermediate"] = intermediate

    # Validate the public intermediate columns that are exposed in the release CSV.
    assert_close(coverage, row["ct_hetero_conversion_fraction_positive"], f"CRC coverage {row['source_patient_id']}")
    assert_close(mean_support, row["ct_hetero_conversion_mean"], f"CRC mean {row['source_patient_id']}")
    assert_close(p10_support, row["ct_hetero_conversion_p10"], f"CRC p10 {row['source_patient_id']}")
    assert_close(effective_coverage, row["module_coverage_effective"], f"CRC effective coverage {row['source_patient_id']}")

# Second pass: rank the label-free intermediate within the 11-patient panel.
intermediates = [row["recomputed_module_calibrated_intermediate"] for row in crc_rows]
ranks = rankdata(intermediates)
for row, rank in zip(crc_rows, ranks):
    recomputed_rank_score = rank / len(crc_rows)
    row["recomputed_response_score_rank_calibrated"] = recomputed_rank_score
    row["recomputed_predicted_responder_rank_ge_0p5"] = recomputed_rank_score >= 0.5
    assert_close(recomputed_rank_score, row["response_score_rank_calibrated"], f"CRC rank {row['source_patient_id']}")

# Third pass: evaluate against clinical response labels.
crc_labels = [int(float(row["observed_responder"])) for row in crc_rows]
crc_scores = [row["recomputed_response_score_rank_calibrated"] for row in crc_rows]
crc_predictions = [score >= 0.5 for score in crc_scores]
crc_pr_scores = [score for label, score in zip(crc_labels, crc_scores) if label == 1]
crc_sd_scores = [score for label, score in zip(crc_labels, crc_scores) if label == 0]
crc_metric = {
    "score_col": "response_score_rank_calibrated",
    "n": len(crc_rows),
    "n_pr": len(crc_pr_scores),
    "n_sd": len(crc_sd_scores),
    "auc_response_high": roc_auc(crc_labels, crc_scores),
    "fixed_0p5_balanced_accuracy": balanced_accuracy(crc_labels, crc_predictions),
    "predicted_responder_count_fixed_0p5": sum(crc_predictions),
    "mean_pr": mean(crc_pr_scores),
    "mean_sd": mean(crc_sd_scores),
    "pr_minus_sd_mean": mean(crc_pr_scores) - mean(crc_sd_scores),
}

# Validate against the checked-in metric CSV.
expected_crc = read_csv_rows(CRC_METRICS_CSV)[0]
for key in [
    "n", "n_pr", "n_sd", "auc_response_high", "fixed_0p5_balanced_accuracy",
    "predicted_responder_count_fixed_0p5", "mean_pr", "mean_sd", "pr_minus_sd_mean",
]:
    assert_close(crc_metric[key], expected_crc[key], f"CRC metric {key}")

print(json.dumps(crc_metric, indent=2, sort_keys=True))

## cSCC Checkpoint Compartment Score

The cSCC checkpoint score is the product of five released axis supports. It keeps the universal response-axis idea that all response gates must pass, while routing axes to the promoted checkpoint compartments.

```text
relative_response_probability =
  engagement_support
  * response_conversion_support
  * coverage_support
  * resistant_tail_control
  * escape_refuge_control
```

Labels are used only after score assignment to compute AUC and Spearman.

In [ ]:
# -----------------------------------------------------------------------------
# Benchmark 2: cSCC checkpoint compartment patient score
# -----------------------------------------------------------------------------
# Input artifact:
#   patient-level-bench/model_scores/cscc_checkpoint_compartment_20260604/
#   cscc_checkpoint_compartment_patient_scores_20260604.csv
#
# Label column:
#   response_label
#
# Public score column:
#   relative_response_probability
#
# Score formula:
#   product of engagement, response conversion, coverage, resistant-tail
#   control, and escape/refuge control supports.
# -----------------------------------------------------------------------------

cscc_rows = read_csv_rows(CSCC_ROWS_CSV)
CSCC_AXIS_COLS = [
    "engagement_support",
    "response_conversion_support",
    "coverage_support",
    "resistant_tail_control",
    "escape_refuge_control",
]

for row in cscc_rows:
    # Compute the exact product score from the five public axis columns.
    axis_values = [finite_float(row[col]) for col in CSCC_AXIS_COLS]
    product_score = 1.0
    for value in axis_values:
        product_score *= value

    # Also recompute the two helper probability summaries included in the CSV.
    geomean_score = product_score ** (1.0 / len(axis_values))
    softmin_score = min(axis_values)

    row["recomputed_relative_response_probability"] = product_score
    row["recomputed_nonresponse_probability"] = 1.0 - product_score
    row["recomputed_universal_axis_geomean_response_probability"] = geomean_score
    row["recomputed_universal_axis_softmin_response_probability"] = softmin_score

    # Validate the row-level public score columns.
    assert_close(product_score, row["relative_response_probability"], f"cSCC product {row['patient_id']}")
    assert_close(1.0 - product_score, row["nonresponse_probability"], f"cSCC nonresponse {row['patient_id']}")
    assert_close(geomean_score, row["universal_axis_geomean_response_probability"], f"cSCC geomean {row['patient_id']}")
    assert_close(softmin_score, row["universal_axis_softmin_response_probability"], f"cSCC softmin {row['patient_id']}")

# Evaluate against response labels.
cscc_labels = [int(float(row["response_label"])) for row in cscc_rows]
cscc_scores = [row["recomputed_relative_response_probability"] for row in cscc_rows]
cscc_response_scores = [score for label, score in zip(cscc_labels, cscc_scores) if label == 1]
cscc_nonresponse_scores = [score for label, score in zip(cscc_labels, cscc_scores) if label == 0]
cscc_metric = {
    "axis_variant": cscc_rows[0]["axis_variant"],
    "mode": cscc_rows[0]["mode"],
    "cascade_step": cscc_rows[0]["cascade_step"],
    "n_patients": len(cscc_rows),
    "n_responders": len(cscc_response_scores),
    "n_nonresponders": len(cscc_nonresponse_scores),
    "auc_response_high": roc_auc(cscc_labels, cscc_scores),
    "spearman_response_high": spearman(cscc_labels, cscc_scores),
    "response_mean": mean(cscc_response_scores),
    "nonresponse_mean": mean(cscc_nonresponse_scores),
    "response_minus_nonresponse": mean(cscc_response_scores) - mean(cscc_nonresponse_scores),
    "mean_probability": mean(cscc_scores),
}

# Validate against the checked-in metric CSV.
expected_cscc = read_csv_rows(CSCC_METRICS_CSV)[0]
for key in [
    "n_patients", "n_responders", "n_nonresponders", "auc_response_high",
    "spearman_response_high", "response_mean", "nonresponse_mean",
    "response_minus_nonresponse", "mean_probability",
]:
    assert_close(cscc_metric[key], expected_cscc[key], f"cSCC metric {key}")

print(json.dumps(cscc_metric, indent=2, sort_keys=True))

## CRC Module Mean Cosine Readout

The CRC cosine readout asks whether predicted tumor program deltas align with measured tumor on-treatment minus pretreatment deltas.

For each patient and cascade step:

```text
module_mean_cosine = cosine(
  predicted module-mean delta vector,
  observed module-mean delta vector
)
```

This is an observed alignment readout, not the pretreatment response rank benchmark.

In [ ]:
# -----------------------------------------------------------------------------
# Readout 3: CRC module mean cosine trajectory
# -----------------------------------------------------------------------------
# Input artifact:
#   patient-level-bench/observed_readouts/crc_module_mean_cosine_20260604/
#   crc_module_mean_cosine_module_vectors.csv
#
# This is an observed on-treatment alignment readout. It is not the CRC
# pretreatment response classifier.
#
# Metric formula:
#   cosine(predicted module-mean delta vector, observed module-mean delta vector)
# -----------------------------------------------------------------------------

module_vector_rows = read_csv_rows(COSINE_MODULE_VECTORS_CSV)

# Group module vectors by cascade step and patient.
by_patient_step = {}
for row in module_vector_rows:
    key = (row["cascade_step"], row["source_patient_id"])
    by_patient_step.setdefault(key, []).append(row)

# Recompute per-patient module mean cosine from module-vector rows.
patient_step_rows = []
for (step, patient_id), rows in sorted(by_patient_step.items(), key=lambda item: (int(item[0][0]), item[0][1])):
    predicted = [finite_float(row["module_predicted_delta"]) for row in rows]
    observed = [finite_float(row["module_observed_delta"]) for row in rows]
    score = cosine(predicted, observed)
    first = rows[0]
    patient_step_rows.append({
        "cascade_step": int(step),
        "source_patient_id": patient_id,
        "clinical_outcome": first["clinical_outcome"],
        "regimen": first["regimen"],
        "n_genes": 41,
        "n_modules": len({row["module"] for row in rows}),
        "module_mean_cosine": score,
    })

# Validate per-patient step scores against the checked-in artifact.
expected_patient_steps = {
    (row["cascade_step"], row["source_patient_id"]): row
    for row in read_csv_rows(COSINE_PATIENT_STEPS_CSV)
}
for row in patient_step_rows:
    expected = expected_patient_steps[(str(row["cascade_step"]), row["source_patient_id"])]
    for key in ["n_genes", "n_modules", "module_mean_cosine"]:
        assert_close(row[key], expected[key], f"cosine patient {row['cascade_step']} {row['source_patient_id']} {key}")

# Summarize each cascade step across patients.
def is_partial_response(row):
    return str(row["clinical_outcome"]).strip().lower() == "partial response"

step_summary = []
for step in sorted({row["cascade_step"] for row in patient_step_rows}):
    rows = [row for row in patient_step_rows if row["cascade_step"] == step]
    scores = [row["module_mean_cosine"] for row in rows]
    pr_scores = [row["module_mean_cosine"] for row in rows if is_partial_response(row)]
    sd_scores = [row["module_mean_cosine"] for row in rows if not is_partial_response(row)]
    labels = [int(is_partial_response(row)) for row in rows]
    highest = max(rows, key=lambda row: row["module_mean_cosine"])
    step_summary.append({
        "cascade_step": step,
        "n_patients": len(rows),
        "mean": mean(scores),
        "median": quantile(scores, 0.50),
        "q25": quantile(scores, 0.25),
        "q75": quantile(scores, 0.75),
        "pr_mean": mean(pr_scores),
        "sd_mean": mean(sd_scores),
        "pr_high_auc": roc_auc(labels, scores),
        "highest_patient": highest["source_patient_id"],
        "highest_outcome": highest["clinical_outcome"],
        "highest_value": highest["module_mean_cosine"],
    })

# Validate step summaries against the checked-in artifact.
expected_step_summary = {int(row["cascade_step"]): row for row in read_csv_rows(COSINE_STEP_SUMMARY_CSV)}
for row in step_summary:
    expected = expected_step_summary[row["cascade_step"]]
    for key in ["n_patients", "mean", "median", "q25", "q75", "pr_mean", "sd_mean", "pr_high_auc", "highest_value"]:
        assert_close(row[key], expected[key], f"cosine step {row['cascade_step']} {key}")
    assert row["highest_patient"] == expected["highest_patient"]
    assert row["highest_outcome"] == expected["highest_outcome"]

best_step = max(step_summary, key=lambda row: row["mean"])
print(json.dumps(best_step, indent=2, sort_keys=True))

## Headline Numbers

These are the patient-level numbers we present publicly. They are derived above from row-level artifacts.

In [ ]:
# -----------------------------------------------------------------------------
# Patient headline table
# -----------------------------------------------------------------------------
# These are the numbers we quote in the README/blog. They are pulled from the
# recomputed metric dictionaries above, not read from a precomputed summary.
# -----------------------------------------------------------------------------

patient_headlines = {
    "CRC rows": crc_metric["n"],
    "CRC AUC response high": round(crc_metric["auc_response_high"], 3),
    "CRC fixed 0.5 balanced accuracy": round(crc_metric["fixed_0p5_balanced_accuracy"], 3),
    "cSCC rows": cscc_metric["n_patients"],
    "cSCC AUC response high": round(cscc_metric["auc_response_high"], 3),
    "cSCC Spearman response high": round(cscc_metric["spearman_response_high"], 3),
    "CRC cosine best step": best_step["cascade_step"],
    "CRC cosine step 1 mean": round(step_summary[0]["mean"], 3),
    "CRC cosine step 4 mean": round(best_step["mean"], 3),
    "CRC cosine step 4 PR mean": round(best_step["pr_mean"], 3),
    "CRC cosine step 4 SD mean": round(best_step["sd_mean"], 3),
}
patient_headlines


## Figure Code

The next cells generate the patient-level figures from the same row-level data. These are intentionally simple SVGs so the figure generation is visible and reproducible without extra plotting packages.

In [ ]:
# -----------------------------------------------------------------------------
# Patient figure helpers
# -----------------------------------------------------------------------------
# The plotting code below is plain SVG generation. It avoids extra plotting
# dependencies and makes every plotted bar/line traceable to the row-level
# benchmark artifacts loaded above.
# -----------------------------------------------------------------------------

def write_svg(path, body, width=760, height=520):
    """Write a complete SVG file from an SVG body string."""
    svg = (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" '
        f'viewBox="0 0 {width} {height}">{body}</svg>'
    )
    path.write_text(svg + "\\n", encoding="utf-8")
    return path


def show_svg(path):
    """Display SVG inline in Jupyter, or print the path in plain Python."""
    if SVG is None:
        print(path)
    else:
        display(SVG(filename=str(path)))


def patient_bar_chart(rows, score_col, label_col, title, output_name, y_max=None):
    """Plot patient scores as bars colored by observed responder label."""
    width, height = 840, 500
    left, right, top, bottom = 72, 26, 58, 96
    plot_w = width - left - right
    plot_h = height - top - bottom
    values = [finite_float(row.get(score_col)) for row in rows]
    y_max = y_max or max(values) * 1.15 or 1.0
    bar_w = plot_w / max(1, len(rows)) * 0.72

    def sy(value):
        return top + plot_h * (1 - value / y_max)

    body = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{left}" y="32" font-family="Inter, Arial" font-size="22" font-weight="700" fill="#202124">{html.escape(title)}</text>',
        f'<line x1="{left}" y1="{top + plot_h}" x2="{left + plot_w}" y2="{top + plot_h}" stroke="#333"/>',
        f'<line x1="{left}" y1="{top}" x2="{left}" y2="{top + plot_h}" stroke="#333"/>',
    ]

    # Grid lines and y-axis labels.
    for tick_i in range(6):
        tick = y_max * tick_i / 5
        y = sy(tick)
        body.append(f'<line x1="{left}" y1="{y:.1f}" x2="{left + plot_w}" y2="{y:.1f}" stroke="#eef0f2"/>')
        body.append(f'<text x="{left - 10}" y="{y + 4:.1f}" text-anchor="end" font-family="Inter, Arial" font-size="12" fill="#5f6368">{tick:.2f}</text>')

    # For rank-score plots, show the fixed 0.5 decision threshold.
    if y_max >= 0.5:
        y = sy(0.5)
        body.append(f'<line x1="{left}" y1="{y:.1f}" x2="{left + plot_w}" y2="{y:.1f}" stroke="#9aa0a6" stroke-dasharray="5 5"/>')

    # Draw one bar per patient.
    for i, row in enumerate(rows):
        value = finite_float(row.get(score_col))
        is_response = str(row.get(label_col)).strip().lower() in {
            "1", "true", "responder", "partial response", "pcr", "mpr"
        }
        color = "#2f6f9f" if is_response else "#b84a3a"
        x = left + plot_w * (i + 0.5) / len(rows) - bar_w / 2
        y = sy(value)
        body.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_w:.1f}" height="{top + plot_h - y:.1f}" fill="{color}"/>')
        label = row.get("source_patient_id") or row.get("patient_id") or str(i + 1)
        body.append(
            f'<text x="{x + bar_w / 2:.1f}" y="{height - 58}" text-anchor="middle" '
            f'font-family="Inter, Arial" font-size="11" fill="#333" '
            f'transform="rotate(-45 {x + bar_w / 2:.1f} {height - 58})">{html.escape(str(label))}</text>'
        )

    # Legend.
    body.append(f'<rect x="{left}" y="{height - 20}" width="12" height="12" fill="#2f6f9f"/>')
    body.append(f'<text x="{left + 18}" y="{height - 10}" font-family="Inter, Arial" font-size="12" fill="#333">Responder</text>')
    body.append(f'<rect x="{left + 116}" y="{height - 20}" width="12" height="12" fill="#b84a3a"/>')
    body.append(f'<text x="{left + 134}" y="{height - 10}" font-family="Inter, Arial" font-size="12" fill="#333">Non-responder</text>')

    return write_svg(FIG_DIR / output_name, "".join(body), width, height)


def cosine_trajectory_chart(step_rows, output_name):
    """Plot all-patient, PR, and SD module-mean cosine across cascade steps."""
    width, height = 820, 500
    left, right, top, bottom = 74, 34, 58, 70
    plot_w = width - left - right
    plot_h = height - top - bottom
    y_min, y_max = -0.25, 0.60

    def sx(step):
        return left + plot_w * (step - 1) / 7

    def sy(value):
        return top + plot_h * (1 - (value - y_min) / (y_max - y_min))

    body = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{left}" y="32" font-family="Inter, Arial" font-size="22" font-weight="700" fill="#202124">CRC Module Mean Cosine by Cascade Step</text>',
        f'<line x1="{left}" y1="{sy(0):.1f}" x2="{left + plot_w}" y2="{sy(0):.1f}" stroke="#333"/>',
        f'<line x1="{left}" y1="{top}" x2="{left}" y2="{top + plot_h}" stroke="#333"/>',
    ]

    # Grid lines.
    for tick in [-0.2, 0.0, 0.2, 0.4, 0.6]:
        y = sy(tick)
        body.append(f'<line x1="{left}" y1="{y:.1f}" x2="{left + plot_w}" y2="{y:.1f}" stroke="#eef0f2"/>')
        body.append(f'<text x="{left - 10}" y="{y + 4:.1f}" text-anchor="end" font-family="Inter, Arial" font-size="12" fill="#5f6368">{tick:.1f}</text>')

    # Draw three line series: all patients, PR patients, and SD patients.
    series = [("mean", "All", "#202124"), ("pr_mean", "PR", "#2f6f9f"), ("sd_mean", "SD", "#b84a3a")]
    for key, label, color in series:
        points = []
        for row in step_rows:
            step = int(row["cascade_step"])
            value = finite_float(row[key])
            points.append(f'{sx(step):.1f},{sy(value):.1f}')
        point_text = " ".join(points)
        body.append(f'<polyline points="{point_text}" fill="none" stroke="{color}" stroke-width="3"/>')
        for row in step_rows:
            step = int(row["cascade_step"])
            value = finite_float(row[key])
            body.append(f'<circle cx="{sx(step):.1f}" cy="{sy(value):.1f}" r="4" fill="{color}"/>')

    # Step labels and legend.
    for step in range(1, 9):
        body.append(f'<text x="{sx(step):.1f}" y="{height - 36}" text-anchor="middle" font-family="Inter, Arial" font-size="12" fill="#333">{step}</text>')
    for i, (_, label, color) in enumerate(series):
        x = left + i * 84
        body.append(f'<line x1="{x}" y1="{height - 16}" x2="{x + 18}" y2="{height - 16}" stroke="{color}" stroke-width="3"/>')
        body.append(f'<text x="{x + 24}" y="{height - 12}" font-family="Inter, Arial" font-size="12" fill="#333">{label}</text>')

    return write_svg(FIG_DIR / output_name, "".join(body), width, height)

print("Figure helpers loaded.")


In [ ]:
# -----------------------------------------------------------------------------
# Generate patient figures
# -----------------------------------------------------------------------------
# Figures written here are reproducible notebook outputs. They are intentionally
# written under artifacts/notebook_figures/, which is gitignored.
# -----------------------------------------------------------------------------

# Sort patients by score so the bars are readable.
crc_plot_rows = sorted(
    crc_rows,
    key=lambda row: finite_float(row["recomputed_response_score_rank_calibrated"]),
    reverse=True,
)
cscc_plot_rows = sorted(
    cscc_rows,
    key=lambda row: finite_float(row["recomputed_relative_response_probability"]),
    reverse=True,
)

# Generate every patient-level figure used by the release notebook.
figures = []
figures.append(patient_bar_chart(
    crc_plot_rows,
    "recomputed_response_score_rank_calibrated",
    "observed_responder",
    "CRC Patient Response Rank Score",
    "patient_crc_rank_score.svg",
    y_max=1.0,
))
figures.append(patient_bar_chart(
    cscc_plot_rows,
    "recomputed_relative_response_probability",
    "response_label",
    "cSCC Checkpoint Compartment Response Score",
    "patient_cscc_checkpoint_score.svg",
))
figures.append(cosine_trajectory_chart(
    step_summary,
    "patient_crc_module_mean_cosine_trajectory.svg",
))

for figure in figures:
    show_svg(figure)

[str(figure.relative_to(ROOT)) for figure in figures]
